# Modelos ARIMA y SARIMA

La selección combina evidencia de ACF y PACF, una rejilla acotada y diagnóstico de residuos. AIC y BIC se comparan únicamente entre modelos ARIMA ajustados a la misma serie.

## Parámetros iniciales

En total, vía aérea y vía terrestre, la ACF en niveles decae lentamente y la PACF concentra señal en los primeros rezagos, por lo que se exploran p y q entre 0 y 2, con d=1. Los picos anuales justifican D=1 y componentes P y Q entre 0 y 1. El mismo rango se usa para El Salvador, Estados Unidos y Honduras, cuyas ACF diferenciadas conservan dependencia corta y señal en el rezago 12. Marítima requiere d=2 como escenario exploratorio porque ninguna transformación probada rechaza raíz unitaria; D=1 se conserva para capturar el ciclo anual, sin presentar la serie como estacionaria. La rejilla permite que AIC decida entre términos autorregresivos y de media móvil dentro de esos rangos, mientras Ljung-Box comprueba si queda autocorrelación residual.

In [1]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src.modelos import (
    ajustar_sarima,
    diagnostico_residuos,
    grid_sarima,
    seleccionar_sarima,
)
from src.utils import RUTA_FIGURAS, RUTA_RESULTADOS, SERIES, cargar_serie

PARAMETROS = {
    clave: {"d": 2 if clave == "via_maritima" else 1, "D": 1}
    for clave in SERIES
}
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

In [2]:
def ejecutar_auto_arima(serie_log, d, D):
    try:
        from pmdarima import auto_arima

        modelo = auto_arima(
            serie_log,
            seasonal=True,
            m=12,
            d=d,
            D=D,
            start_p=0,
            start_q=0,
            max_p=2,
            max_q=2,
            start_P=0,
            start_Q=0,
            max_P=1,
            max_Q=1,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore",
        )
        return modelo.order, modelo.seasonal_order, modelo.aic()
    except Exception as error:
        return None, None, f"No disponible: {error}"


comparaciones = []
automaticos = []
for clave in SERIES:
    serie_log = np.log1p(cargar_serie(clave, "train"))
    parametros = PARAMETROS[clave]
    rejilla = grid_sarima(serie_log, horizonte=63, **parametros)
    validos = rejilla[np.isfinite(rejilla["aic"])]
    estables = validos[validos["estable"]]
    elegida = seleccionar_sarima(rejilla)
    candidatos = estables.head(3) if len(estables) >= 3 else validos.head(3)

    print(f"\n{SERIES[clave]}: top 5 por AIC con filtro de estabilidad")
    print(
        validos[
            ["order", "seasonal_order", "aic", "bic", "estable", "max_pronostico"]
        ]
        .head(5)
        .to_string(index=False)
    )
    print(
        f"ajustados={len(validos)} estables={len(estables)} "
        f"elegida={elegida['order']}{elegida['seasonal_order']} "
        f"estable={bool(elegida['estable'])}"
    )

    auto_order, auto_seasonal, auto_aic = ejecutar_auto_arima(
        serie_log,
        parametros["d"],
        parametros["D"],
    )
    automaticos.append(
        {
            "serie": clave,
            "order": auto_order,
            "seasonal_order": auto_seasonal,
            "aic": auto_aic,
        }
    )
    print(
        "auto_arima:",
        auto_order,
        auto_seasonal,
        auto_aic,
    )

    for _, fila in candidatos.reset_index(drop=True).iterrows():
        order = tuple(fila["order"])
        seasonal_order = tuple(fila["seasonal_order"])
        modelo = ajustar_sarima(serie_log, order, seasonal_order)
        residuos, figura = diagnostico_residuos(modelo)
        seleccionado = (
            order == tuple(elegida["order"])
            and seasonal_order == tuple(elegida["seasonal_order"])
        )
        if seleccionado:
            ruta = RUTA_FIGURAS / f"modelo_{clave}_residuos.png"
            figura.savefig(ruta, dpi=150, bbox_inches="tight")
        plt.close(figura)
        comparaciones.append(
            {
                "serie": clave,
                "order": order,
                "seasonal_order": seasonal_order,
                "aic": modelo.aic,
                "bic": modelo.bic,
                "ljung_box_p": residuos["ljung_box_p"],
                "jarque_bera_p": residuos["jarque_bera_p"],
                "max_pronostico": fila["max_pronostico"],
                "estable": bool(fila["estable"]),
                "n_ajustados": len(validos),
                "n_estables": len(estables),
                "seleccionado": seleccionado,
            }
        )

    if not any(
        fila["seleccionado"]
        for fila in comparaciones
        if fila["serie"] == clave
    ):
        modelo = ajustar_sarima(
            serie_log,
            tuple(elegida["order"]),
            tuple(elegida["seasonal_order"]),
        )
        residuos, figura = diagnostico_residuos(modelo)
        ruta = RUTA_FIGURAS / f"modelo_{clave}_residuos.png"
        figura.savefig(ruta, dpi=150, bbox_inches="tight")
        plt.close(figura)
        comparaciones.append(
            {
                "serie": clave,
                "order": tuple(elegida["order"]),
                "seasonal_order": tuple(elegida["seasonal_order"]),
                "aic": modelo.aic,
                "bic": modelo.bic,
                "ljung_box_p": residuos["ljung_box_p"],
                "jarque_bera_p": residuos["jarque_bera_p"],
                "max_pronostico": elegida["max_pronostico"],
                "estable": bool(elegida["estable"]),
                "n_ajustados": len(validos),
                "n_estables": len(estables),
                "seleccionado": True,
            }
        )

modelos_arima = pd.DataFrame(comparaciones)
modelos_arima.to_csv(RUTA_RESULTADOS / "modelos_arima.csv", index=False)
auto_arima_resultados = pd.DataFrame(automaticos)
print("\nComparación manual consolidada")
print(modelos_arima.to_string(index=False))
print("\nResultados de auto_arima")
print(auto_arima_resultados.to_string(index=False))


Total: top 5 por AIC con filtro de estabilidad
    order seasonal_order       aic       bic  estable  max_pronostico
(2, 1, 2)  (1, 1, 1, 12) 67.955300 87.409164     True   114117.487817
(1, 1, 2)  (1, 1, 1, 12) 68.547485 85.222226     True   110457.238105
(2, 1, 2)  (1, 1, 0, 12) 68.962388 85.687339     True   116226.339725
(2, 1, 2)  (0, 1, 1, 12) 69.193863 85.868604     True   102973.961175
(1, 1, 2)  (0, 1, 1, 12) 69.447859 83.343477     True   102780.943721
ajustados=36 estables=36 elegida=(2, 1, 2)(1, 1, 1, 12) estable=True


auto_arima: (0, 1, 0) (0, 1, 1, 12) 73.96452580289909


/home/escu/Documentos/Universidad/Semestres/8voSemestre/DATA_SCIENCE/DATA_SCIENCE/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


/home/escu/Documentos/Universidad/Semestres/8voSemestre/DATA_SCIENCE/DATA_SCIENCE/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



Vía Aérea: top 5 por AIC con filtro de estabilidad
    order seasonal_order        aic        bic  estable  max_pronostico
(2, 1, 2)  (0, 1, 1, 12) 169.721392 186.396133     True    63866.974167
(0, 1, 1)  (0, 1, 1, 12) 176.288057 184.650532     True    81343.220586
(0, 1, 2)  (0, 1, 1, 12) 176.979172 188.095666     True    77031.134903
(1, 1, 0)  (0, 1, 1, 12) 177.333027 185.720398     True    79853.699774
(0, 1, 0)  (0, 1, 1, 12) 177.429378 183.020959     True    69264.850426
ajustados=36 estables=18 elegida=(2, 1, 2)(0, 1, 1, 12) estable=True


...

/home/escu/Documentos/Universidad/Semestres/8voSemestre/DATA_SCIENCE/DATA_SCIENCE/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



Honduras: top 5 por AIC con filtro de estabilidad
    order seasonal_order        aic        bic  estable  max_pronostico
(2, 1, 2)  (1, 1, 0, 12) 359.014701 375.739651     True    11521.854962
(1, 1, 2)  (0, 1, 1, 12) 360.205135 374.100753     True    34469.056153
(0, 1, 2)  (0, 1, 1, 12) 370.319320 381.435814     True     8885.169194
(0, 1, 1)  (0, 1, 1, 12) 370.456983 378.819458     True     8868.746401
(0, 1, 0)  (0, 1, 1, 12) 370.488508 376.080089     True     8776.200583
ajustados=36 estables=13 elegida=(2, 1, 2)(1, 1, 0, 12) estable=True


auto_arima: (2, 1, 1) (0, 1, 1, 12) 395.1913944523051


/home/escu/Documentos/Universidad/Semestres/8voSemestre/DATA_SCIENCE/DATA_SCIENCE/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



Comparación manual consolidada
              serie     order seasonal_order        aic        bic  ljung_box_p  jarque_bera_p  max_pronostico  estable  n_ajustados  n_estables  seleccionado
              total (2, 1, 2)  (1, 1, 1, 12)  67.955300  87.409164 2.640950e-04   0.000000e+00   114117.487817     True           36          36          True
              total (1, 1, 2)  (1, 1, 1, 12)  68.547485  85.222226 4.570340e-03   0.000000e+00   110457.238105     True           36          36         False
              total (2, 1, 2)  (1, 1, 0, 12)  68.962388  85.687339 1.159645e-03   0.000000e+00   116226.339725     True           36          36         False
          via_aerea (2, 1, 2)  (0, 1, 1, 12) 169.721392 186.396133 1.908017e-07   0.000000e+00    63866.974167     True           36          18          True
          via_aerea (0, 1, 1)  (0, 1, 1, 12) 176.288057 184.650532 6.012142e-03   0.000000e+00    81343.220586     True           36          18         False
          via_

## Criterio de selección

Para cada serie se retienen tres especificaciones convergentes con menor AIC y se comparan también por BIC, Ljung-Box y Jarque-Bera. `auto_arima` se restringe al mismo espacio de búsqueda, así que una propuesta con términos de orden bajo y diferenciación fijada es coherente con las ACF y PACF observadas. Si difiere del mínimo manual, la causa puede ser su búsqueda escalonada y no una contradicción con el diagnóstico. Si la biblioteca no carga por incompatibilidad binaria, la salida de la celda registra el error y la rejilla manual queda como método reproducible. Jarque-Bera se informa como diagnóstico, pero la falta de normalidad no invalida por sí sola el pronóstico; la ausencia de autocorrelación residual tiene mayor peso.

## Modelos alternativos

Se generan cuatro referencias con el mismo horizonte del test. Holt-Winters y suavizamiento exponencial simple se ajustan en `log1p` y se revierten a viajeros; seasonal naive trabaja en niveles y repite los últimos doce meses; Prophet usa estacionalidad anual sobre `log1p`. Si Prophet no está disponible, el error queda registrado y los otros tres modelos continúan.

In [3]:
from src.modelos import (
    modelo_holt_winters,
    modelo_prophet,
    modelo_seasonal_naive,
    modelo_ses,
)

RUTA_PREDICCIONES = RUTA_RESULTADOS / "predicciones"
RUTA_PREDICCIONES.mkdir(parents=True, exist_ok=True)
MODELOS_ALTERNATIVOS = {
    "holt_winters": modelo_holt_winters,
    "ses": modelo_ses,
    "seasonal_naive": modelo_seasonal_naive,
    "prophet": modelo_prophet,
}

def guardar_prediccion(clave, modelo, prediccion):
    salida = pd.DataFrame(
        {
            "fecha": prediccion.index.strftime("%Y-%m-%d"),
            "prediccion": prediccion.to_numpy(dtype=float),
        }
    )
    salida.to_csv(RUTA_PREDICCIONES / f"{clave}_{modelo}.csv", index=False)


for clave in SERIES:
    train = cargar_serie(clave, "train")
    horizonte = len(cargar_serie(clave, "test"))
    for nombre, funcion in MODELOS_ALTERNATIVOS.items():
        try:
            prediccion = funcion(train, horizonte)
            guardar_prediccion(clave, nombre, prediccion)
            print(clave, nombre, len(prediccion))
        except Exception as error:
            print(clave, nombre, f"no disponible: {error}")

total holt_winters 63
total ses 63
total seasonal_naive 63


Importing plotly failed. Interactive plots will not work.


15:50:46 - cmdstanpy - INFO - Chain [1] start processing


15:50:46 - cmdstanpy - INFO - Chain [1] done processing


15:50:46 - cmdstanpy - INFO - Chain [1] start processing


...

15:50:47 - cmdstanpy - INFO - Chain [1] done processing


pais_el_salvador holt_winters 63
pais_el_salvador ses 63
pais_el_salvador seasonal_naive 63
pais_el_salvador prophet 63
pais_estados_unidos holt_winters 63
pais_estados_unidos ses 63
pais_estados_unidos seasonal_naive 63


15:50:47 - cmdstanpy - INFO - Chain [1] start processing


15:50:47 - cmdstanpy - INFO - Chain [1] done processing


pais_estados_unidos prophet 63
pais_honduras holt_winters 63
pais_honduras ses 63
pais_honduras seasonal_naive 63
pais_honduras prophet 63
